Raw text - len 5572 <br>
   ↓  tokenize (split into words)<br>
   ↓  convert to IDs (vocab lookup)<br>
   ↓  pad sequences (make equal length) - [5572,216], <br>
   ↓  nn.Embedding (row lookup → vectors) [5572,216,128],embed size = 128<br>
   ↓  nn.RNN  - input - [5572,216,128] , hidden_state = 3, input_feature = 128, h_final = [layer,batch_size,hidden], layer = 1, <br>h_final = [1,5572,3],squeeze,h_final=[5572,3]<br>
   ↓  nn.Linear - in_features=3,out_features=2 - [5572,2] - apply softmax along dim=0 - argmax<br>
   ↓  spam or ham <br>

In [1]:
import torch
import pandas as pd
import nltk
from nltk.tokenize import word_tokenize
file_name = r"/Users/sanvijanvi/Library/Mobile Documents/com~apple~CloudDocs/ML_DL_Projects/NLP/SMSSpamCollection.txt"

messages = pd.read_csv(file_name,sep="\t",names=["label","message"])

print(len(messages))

all_tokens = []
for msg in messages["message"]:
    # print("msg:",msg)
    tokens = word_tokenize(msg.lower())
    # print("msg:",tokens)
    all_tokens.extend(tokens)

print(all_tokens.__contains__("joking"))

vocab = {"<pad>":0,"<unk>":1}
for token in set(all_tokens):
    if token not in vocab:
        vocab[token] = len(vocab)

print(vocab)



5572
True
{'<pad>': 0, '<unk>': 1, 'us.get': 2, '09058098002.': 3, 'bcz': 4, 'mundhe': 5, 'kid': 6, 'mummy': 7, 'australia': 8, 'boys': 9, 'hogli': 10, 'yetunde': 11, 'decking': 12, 'shijutta': 13, 'salad': 14, 'andre': 15, 'pehle': 16, '4.15': 17, 'scammers': 18, 'child': 19, 'oso': 20, 'untamed': 21, 'whilltake': 22, 'online': 23, 'ups': 24, 'acting': 25, 'works': 26, 'hottest': 27, 'jacuzzi': 28, 'howard': 29, '08718711108': 30, 'if': 31, 'ken': 32, 'worst': 33, '4-6': 34, 'platt': 35, 'nd': 36, 'deepest': 37, '09058094454': 38, 'footbl': 39, 'first': 40, 'pain': 41, 'pics': 42, 'fun': 43, 'iknow': 44, 'bffs': 45, '391784.': 46, 'fails': 47, 'damn': 48, 'tescos': 49, 'conclusion': 50, 'thasa': 51, 'experiencehttp': 52, '08704439680ts': 53, 'acted': 54, 'tenerife': 55, 'car.so': 56, 'penis': 57, 'market': 58, 'grinder': 59, 'ground.amla': 60, 'cappuccino': 61, '9th': 62, 'connected': 63, 'goin2bed': 64, 'polo': 65, 'tue': 66, 'jokes': 67, 'youuuuu': 68, 'tooo': 69, '1winawk': 70, 'pu

In [2]:
# get the index for each sequence
 # get sequence of indexes for each essage

msg_sequence_list =[]


for msg in messages["message"]:
    msg_sequence = []
    tokens = word_tokenize(msg.lower())
    for token in tokens:
        msg_sequence.append(vocab.get(token,1))
    # print(f"{msg}:{msg_sequence}")
    msg_sequence_list.append(torch.tensor(msg_sequence))

print(len(msg_sequence_list))




5572


# Padding 
Padding happens BEFORE Embedding — because:,<br>

Embedding expects integer token IDs as input<br>
All inputs must be same length before feeding to Embedding<br>
So you pad with 0s first, then Embedding converts everything to vectors<br>

216 is the length of the longest message in your dataset (in tokens). pad_sequence by default returns shape <br>(max_seq_len, batch_size), so all shorter messages got padded with zeros up to 216 tokens.



In [3]:
from torch.nn.utils.rnn import pad_sequence

padded_msg = pad_sequence(msg_sequence_list,batch_first=True) #torch.Size([5572, 216]) - 216 is length of longest msg
padded_msg.shape

torch.Size([5572, 216])

In [ ]:
# embedding now
from torch import nn

embed_layer = nn.Embedding(num_embeddings=len(vocab),embedding_dim=128)
embed_msg = embed_layer(padded_msg)
print(embed_msg.shape)



torch.Size([5572, 216, 128])


In [14]:
# rnn layer
from torch import nn
rnn_layer = nn.RNN(batch_first=True,input_size=128,hidden_size=3)

rnn_out,h_final=rnn_layer(embed_msg)
print(h_final.shape) #(1, 1, 3) — h_final: 1 layer, 1 batch, 3 hidden units

linear_layer = nn.Linear(in_features=3,out_features=2)
linear_out = linear_layer(h_final.squeeze())

print(linear_out.shape,linear_out)

# rnn_out,h_final=rnn_layer(embed_msg[:5,:,:])
# print(rnn_out.shape)

torch.Size([1, 5572, 3])
torch.Size([5572, 2]) tensor([[-1.0374,  0.6846],
        [-1.0374,  0.6846],
        [-1.0374,  0.6846],
        ...,
        [-1.0374,  0.6846],
        [-1.0374,  0.6846],
        [-1.0374,  0.6846]], grad_fn=<AddmmBackward0>)


In [6]:
from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder()
encoded = encoder.fit(messages["label"])

transformed_labels = encoded.transform(messages["label"])
encoded.classes_
transformed_labels[:5]



array([0, 0, 1, 0, 0])

In [7]:
linear_out.shape
sft = torch.softmax(linear_out,dim=0)
print(sft)
arg_sft = torch.argmax(sft,dim=0)
arg_sft


tensor([[0.0002, 0.0002],
        [0.0002, 0.0002],
        [0.0002, 0.0002],
        ...,
        [0.0002, 0.0002],
        [0.0002, 0.0002],
        [0.0002, 0.0002]], grad_fn=<SoftmaxBackward0>)


tensor([1579,    0])

In [16]:
print(embed_msg.shape)
lstm = nn.LSTM(input_size=50, hidden_size=3,batch_first=False)
out, (hidden,cell) = lstm(embed_msg)

print(out.shape)
print(hidden.shape)
print(cell.shape)

torch.Size([5572, 216, 128])
torch.Size([5572, 216, 3])
torch.Size([1, 216, 3])
torch.Size([1, 216, 3])
